In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
data_path = "../../data/"

orders_path = os.path.join(data_path, "orders.csv")
order_products_path = os.path.join(data_path, "order_products__prior.csv")

orders = pd.read_csv(orders_path)
order_products = pd.read_csv(order_products_path)

print(f"Orders shape: {orders.shape}")
print(f"Order products shape: {order_products}")

Orders shape: (3421083, 7)
Order products shape:           order_id  product_id  add_to_cart_order  reordered
0                2       33120                  1          1
1                2       28985                  2          1
2                2        9327                  3          0
3                2       45918                  4          1
4                2       30035                  5          0
...            ...         ...                ...        ...
32434484   3421083       39678                  6          1
32434485   3421083       11352                  7          0
32434486   3421083        4600                  8          0
32434487   3421083       24852                  9          1
32434488   3421083        5020                 10          1

[32434489 rows x 4 columns]


In [3]:
merged_df = pd.merge(
    order_products[["order_id", "product_id"]],
    orders[['order_id', 'user_id']],
    on="order_id",
    how="inner"
)

print(f"Merged shape (Order, Product, User) {merged_df.shape}")
merged_df.head()

Merged shape (Order, Product, User) (32434489, 3)


,order_id,product_id,user_id
0,2,33120,202279
1,2,28985,202279
2,2,9327,202279
3,2,45918,202279
4,2,30035,202279


In [4]:
georgios_path = os.path.join(data_path, "Georgios")
os.makedirs(georgios_path, exist_ok=True)

bui = (
    merged_df.groupby(['user_id', 'product_id'])['order_id']
    .nunique()
    .reset_index(name='Bui')
)

bu = (
    merged_df.groupby(['user_id'])['order_id']
    .nunique()
    .reset_index(name='Bu')
)

freq_df = pd.merge(bui, bu, on='user_id', how='left')
freq_df['freq_ui'] = freq_df['Bui'] / freq_df['Bu']

csv_path = os.path.join(georgios_path, "user_product_frequency.csv")
parquet_path = os.path.join(georgios_path, "user_product_frequency.parquet")

freq_df.to_csv(csv_path, index=False)
freq_df.to_parquet(parquet_path, index=False)

print(f"Frequency data (Bui, Bu, freq_ui) saved to: {csv_path}")
print(f"Frequency data (Bui, Bu, freq_ui) saved to: {parquet_path}")

freq_df.head(10)

Frequency data (Bui, Bu, freq_ui) saved to: ../../data/Georgios\user_product_frequency.csv
Frequency data (Bui, Bu, freq_ui) saved to: ../../data/Georgios\user_product_frequency.parquet


,user_id,product_id,Bui,Bu,freq_ui
0,1,196,10,10,1.0
1,1,10258,9,10,0.9
2,1,10326,1,10,0.1
3,1,12427,10,10,1.0
4,1,13032,3,10,0.3
5,1,13176,2,10,0.2
6,1,14084,1,10,0.1
7,1,17122,1,10,0.1
8,1,25133,8,10,0.8
9,1,26088,2,10,0.2
